<a href="https://colab.research.google.com/github/sultanjacob/Applied-Machine-Learning/blob/main/Phase_10_Dimensionality_Reduction/10_PCA_Predictive_Maintenance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 10: Dimensionality Reduction (PCA)
## Step 1: Data Acquisition and Initial Examination

To demonstrate Principal Component Analysis (PCA), we are using the **SECOM (Semiconductor Manufacturing) Dataset** from the UCI Machine Learning Repository.

This dataset tracks the production of semiconductor wafers. It contains 590 unnamed sensor readings (`Feature_0` to `Feature_589`) and a target variable indicating whether the wafer passed (-1) or failed (1) quality control.

Before we can compress this high-dimensional data, we must first load it into our environment and examine the basic structure and data health (such as missing values) to understand exactly what we are working with.

In [2]:
import pandas as pd
import numpy as np

# 1. Define the direct URLs to the UCI repository
data_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/secom/secom.data"
labels_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/secom/secom_labels.data"

print("⏳ Loading 590-dimensional sensor data...")

# 2. Load the 590 sensor features (space-separated)
X = pd.read_csv(data_url, sep=" ", header=None)
X.columns = [f"Feature_{i}" for i in range(X.shape[1])]

# 3. Load the target labels (-1 = Pass, 1 = Fail)
y = pd.read_csv(labels_url, sep=" ", header=None, usecols=[0])
y.columns = ['Target']

# 4. Combine into a single DataFrame for Exploration
secom_df = pd.concat([X, y], axis=1)

print("✅ Data Loaded Successfully!\n")

# 5. Examine the structural footprint
print("📊 Structural Overview:")
print(f"Total Rows (Batches): {secom_df.shape[0]}")
print(f"Total Columns (Sensors + Target): {secom_df.shape[1]}")

# 6. Examine Data Health (Missing Values)
total_cells = secom_df.size
total_missing = secom_df.isnull().sum().sum()
missing_percentage = (total_missing / total_cells) * 100

print(f"\n⚠️ Data Health Check:")
print(f"Total Missing Values: {total_missing}")
print(f"Percentage of Missing Data: {missing_percentage:.2f}%")

⏳ Loading 590-dimensional sensor data...
✅ Data Loaded Successfully!

📊 Structural Overview:
Total Rows (Batches): 1567
Total Columns (Sensors + Target): 591

⚠️ Data Health Check:
Total Missing Values: 41951
Percentage of Missing Data: 4.53%


## Step 2: Data Cleaning and Variance Filtering

Principal Component Analysis (PCA) relies entirely on calculating the mathematical variance between features. Because of this, it has two strict requirements: it cannot process missing values (`NaN`), and it gains zero information from sensors that never change (zero variance).

With nearly 42,000 missing values scattered across the dataset, we must execute a precise cleaning strategy before scaling:
1. **Heavy Null Removal:** Drop any sensor missing more than 50% of its readings.
2. **Median Imputation:** Fill the remaining missing values with the median of their respective columns. This preserves the baseline signal without letting extreme outliers warp the replacement value.
3. **Constant Feature Removal:** Drop any sensor that outputs the exact same value for every single batch (0 variance), as it provides no predictive power.